In [4]:
import pygame
import numpy as np
import math
import sys

# --- Parâmetros de Simulação e Controle ---
LARGURA_TELA, ALTURA_TELA = 800, 600
ALCANCE_MAXIMO = 150.0  # S_max
ANGULOS_SENSOR = [-60, -30, 0, 30, 60]  # Graus (-esq, +dir)

V_BASE = 100.0          # Velocidade de cruzeiro (px/s)
DIST_EMERGENCIA = 40.0  # Limiar do sensor central
K_S = 1.5               # Ganho de reação

# Cores
BRANCO, PRETO = (255, 255, 255), (0, 0, 0)
AZUL, VERMELHO, VERDE = (50, 150, 255), (255, 50, 50), (50, 255, 50)
CINZA = (100, 100, 100)
COR_TRAJETORIA = (255, 165, 0) # Laranja para destacar o caminho

def intersecao_linhas(p1, p2, p3, p4):
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    x4, y4 = p4
    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if denom == 0: return None
    t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
    u = -((x1 - x2) * (y1 - y3) - (y1 - y2) * (x1 - x3)) / denom
    if 0 <= t <= 1 and 0 <= u <= 1:
        return (x1 + t * (x2 - x1), y1 + t * (y2 - y1))
    return None

def simular():
    pygame.init()
    tela = pygame.display.set_mode((LARGURA_TELA, ALTURA_TELA))
    pygame.display.set_caption("Lab 04 - Braitenberg (Com Trajetória)")
    clock = pygame.time.Clock()

    robo_x, robo_y = 100.0, 100.0
    robo_theta = math.radians(45)
    raio_robo = 15.0
    distancia_rodas = 30.0

    # Lista para armazenar o histórico de posições
    trajetoria = []
    MAX_PONTOS_TRAJETORIA = 800 # Limita o rastro para não sobrecarregar

    obstaculos = [
        pygame.Rect(200, 150, 100, 100),
        pygame.Rect(500, 100, 150, 50),
        pygame.Rect(300, 400, 80, 120),
        pygame.Rect(600, 350, 100, 150),
        pygame.Rect(50, 300, 70, 70)
    ]
    segmentos_paredes = [
        ((0, 0), (LARGURA_TELA, 0)),
        ((LARGURA_TELA, 0), (LARGURA_TELA, ALTURA_TELA)),
        ((LARGURA_TELA, ALTURA_TELA), (0, ALTURA_TELA)),
        ((0, ALTURA_TELA), (0, 0))
    ]
    for obs in obstaculos:
        p1, p2, p3, p4 = obs.topleft, obs.topright, obs.bottomright, obs.bottomleft
        segmentos_paredes.extend([(p1, p2), (p2, p3), (p3, p4), (p4, p1)])

    rodando = True
    while rodando:
        dt = clock.tick(60) / 1000.0
        tela.fill(BRANCO)
        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                rodando = False

        distancias_sensores = []
        for angulo_rel in ANGULOS_SENSOR:
            ang_abs = robo_theta + math.radians(angulo_rel)
            ponto_origem = (robo_x, robo_y)
            ponto_final = (
                robo_x + math.cos(ang_abs) * ALCANCE_MAXIMO,
                robo_y + math.sin(ang_abs) * ALCANCE_MAXIMO
            )
            menor_dist = ALCANCE_MAXIMO
            for seg in segmentos_paredes:
                intersecao = intersecao_linhas(ponto_origem, ponto_final, seg[0], seg[1])
                if intersecao:
                    dist = math.hypot(intersecao[0] - robo_x, intersecao[1] - robo_y)
                    if dist < menor_dist:
                        menor_dist = dist
            
            ruido = np.random.normal(0, 2.0)
            leitura = max(0.0, min(menor_dist + ruido, ALCANCE_MAXIMO))
            distancias_sensores.append(leitura)

            # Renderiza os feixes
            cor = VERMELHO if menor_dist < ALCANCE_MAXIMO else VERDE
            pf_real = (
                robo_x + math.cos(ang_abs) * leitura,
                robo_y + math.sin(ang_abs) * leitura
            )
            pygame.draw.line(tela, cor, ponto_origem, pf_real, 1)

        # Índices: 0 (-60, esq), 1 (-30, esq), 2 (0, centro), 3 (30, dir), 4 (60, dir)
        s_esq = min(distancias_sensores[0], distancias_sensores[1])
        s_dir = min(distancias_sensores[3], distancias_sensores[4])
        s_centro = distancias_sensores[2]

        # Regra 4: v_L = v_base + Ks * (S_max - s_dir)
        #          v_R = v_base + Ks * (S_max - s_esq)
        v_L = V_BASE + K_S * (ALCANCE_MAXIMO - s_dir)
        v_R = V_BASE + K_S * (ALCANCE_MAXIMO - s_esq)

        # Regra 5: Condição de emergência (sensor central < 40 px)
        if s_centro < DIST_EMERGENCIA:
            v_L = -V_BASE  # Inverte a roda esquerda
            v_R = V_BASE   # Mantém a roda direita (giro no próprio eixo)

        # Cinemática do robô diferencial
        v_linear = (v_L + v_R) / 2.0
        v_angular = (v_R - v_L) / distancia_rodas

        robo_theta += v_angular * dt
        robo_x += v_linear * math.cos(robo_theta) * dt
        robo_y += v_linear * math.sin(robo_theta) * dt

        # Mantém na tela
        robo_x = max(raio_robo, min(robo_x, LARGURA_TELA - raio_robo))
        robo_y = max(raio_robo, min(robo_y, ALTURA_TELA - raio_robo))

        # --- ATUALIZA E DESENHA A TRAJETÓRIA ---
        # Adiciona a posição atual (convertida para inteiro) à lista
        trajetoria.append((int(robo_x), int(robo_y)))
        
        # Remove os pontos mais antigos se exceder o limite
        if len(trajetoria) > MAX_PONTOS_TRAJETORIA:
            trajetoria.pop(0)

        # Renderiza os obstáculos
        for obs in obstaculos:
            pygame.draw.rect(tela, CINZA, obs)
            
        # Renderiza a trajetória (precisa de pelo menos 2 pontos para formar uma linha)
        if len(trajetoria) >= 2:
            pygame.draw.lines(tela, COR_TRAJETORIA, False, trajetoria, 2)
        
        # Renderiza o robô
        pygame.draw.circle(tela, AZUL, (int(robo_x), int(robo_y)), int(raio_robo))
        frente_x = robo_x + math.cos(robo_theta) * raio_robo
        frente_y = robo_y + math.sin(robo_theta) * raio_robo
        pygame.draw.line(tela, PRETO, (robo_x, robo_y), (frente_x, frente_y), 3)

        pygame.display.flip()

    pygame.quit()

if __name__ == "__main__":
    try:
        simular()
    except Exception as e:
        print(f"Simulação encerrada ou falhou: {e}")
        pygame.quit()